# Speeding inference using minimal one-vs-one classification models: figures and tables

This notebook recreates the figures and tables of the paper that are computed in R. It runs in an **R kernel** ([IRkernel](https://irkernel.github.io/)); start Jupyter in this folder, so that `R/` and `data/` are found.

* **Section 3** (Figures 3–6, Tables 1–2): simulated examples, rerun here with the seeds of the paper.
* **Sections 5–7** (Figures 7–9, Tables 3, 5 and 6): RBF-SVM experiments on benchmark datasets. They take hours, so the notebook reads their results from `data/`. The code that produced these files is in `experiments/` (see `README.md`).

Figures 1 and 2 (drawings of trees) and Table 4 (overview of the algorithms) are not computed and are not reproduced here.

The notebook runs in about two minutes.

In [ ]:
pkgs <- c("dplyr", "tidyr", "purrr", "ggplot2", "patchwork", "MASS", "caret", "future", "furrr")
for (p in pkgs) if (!requireNamespace(p, quietly = TRUE)) install.packages(p)

In [ ]:
suppressPackageStartupMessages({
  library(dplyr); library(tidyr); library(purrr); library(ggplot2); library(patchwork)
})
source("R/examples.R")          # Example 1: three Gaussian classes
source("R/lda_minimal.R")       # minimal models with LDA pairwise classifiers
source("R/tammes_configs.R")    # optimal Tammes configurations on the sphere
source("R/sim.R")               # von Mises-Fisher classes, spanning trees
source("R/diam3.R")             # diameter-3 trees (double stars)
source("R/plot_helpers.R")      # look of Figures 7-9
options(repr.plot.res = 120)

## Section 3: simulated examples

### Figure 3: the dataset of Example 1

Three classes with bivariate Gaussian distributions, 50 points each. The vertices of the triangle are the class means.

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 4.5)
set.seed(456)
dfs <- dfs_for_r(0, 50, p_train = 1)
dfs$train$class <- dfs$train$class_id

vertices <- data.frame(x = c(0, 0, 4), y = c(0, 3, 0))
sides <- data.frame(x = vertices$x, y = vertices$y,
                    xend = vertices$x[c(2, 3, 1)], yend = vertices$y[c(2, 3, 1)],
                    side = c("12", "23", "13"))
# side labels: at the midpoints, moved outwards along the normal
len <- sqrt((sides$xend - sides$x)^2 + (sides$yend - sides$y)^2)
sides$xo <- (sides$x + sides$xend) / 2 - 0.2 * (sides$yend - sides$y) / len
sides$yo <- (sides$y + sides$yend) / 2 + 0.2 * (sides$xend - sides$x) / len

ggplot() +
  geom_point(data = dfs$train, aes(x = x, y = y, color = class)) +
  theme_minimal(base_size = 10) +
  geom_segment(data = sides, aes(x = x, y = y, xend = xend, yend = yend), linewidth = 0.2,
               arrow = arrow(length = unit(0.25, "cm"), ends = "both", type = "closed")) +
  geom_text(data = sides, aes(x = xo, y = yo, label = side), fontface = "bold", vjust = -0.5) +
  annotate("text", x = 0, y = -0.25, label = expression(bold(mu[1]))) +
  annotate("text", x = 4, y = -0.25, label = expression(bold(mu[3]))) +
  annotate("text", x = 0, y = 3.2, label = expression(bold(mu[2]))) +
  labs(x = expression(f[1]), y = expression(f[2]))

### Figure 4: accuracy of the three minimal models

The minimal models $M1$, $M2$, $M3$ (the stars with centre 1, 2, 3) with LDA pairwise classifiers, 1000 repetitions. Left: accuracy of each model. Right: difference from the accuracy of $M1$ on the same dataset.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 3.2)
exm1 <- lda_example1(r = 0, reps = 1000, n_train = 50, n_test = 150)
exm1_w <- pivot_wider(exm1, names_from = model, values_from = accuracy)
exm1_diff <- rbind(data.frame(model = "M2 - M1", diff = exm1_w$M2 - exm1_w$M1),
                   data.frame(model = "M3 - M1", diff = exm1_w$M3 - exm1_w$M1))
p_acc <- ggplot(exm1, aes(x = model, y = accuracy)) +
  geom_boxplot(outlier.size = 0.5) +
  labs(x = "minimal model", y = "accuracy") + theme_minimal(base_size = 10)
p_diff <- ggplot(exm1_diff, aes(x = model, y = diff)) +
  geom_hline(yintercept = 0, colour = "darkgreen") +
  geom_boxplot(outlier.size = 0.5) +
  labs(x = NULL, y = "difference in accuracy") + theme_minimal(base_size = 10)
p_acc + p_diff

### Figure 5: population Fisher ratios

Population Fisher ratio of each pair of classes as a function of the correlation parameter $r$ of Example 2.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
rs <- seq(-0.8, 0.6, length.out = 50)
fr_full <- map(rs, function(r) {
  v <- compute_population_fisher(r)
  data.frame(v = v, classes = names(v), r = r)
}) |> list_rbind()
# colour and line type: the same as the model omitting the pair in Figure 6
ggplot(fr_full, aes(x = r, y = v, group = classes, color = classes, linetype = classes)) +
  geom_path(linewidth = 0.7) + theme_minimal(base_size = 10) +
  scale_linetype_manual(values = c(`C1 and C2` = "dotdash", `C1 and C3` = "dashed", `C2 and C3` = "solid")) +
  labs(x = expression(plain(correlation)~italic(r)), y = "population Fisher ratio") +
  guides(colour = guide_legend(override.aes = list(linewidth = 0.7))) +
  theme(legend.key.width = grid::unit(1.4, "cm")) +
  scale_x_continuous(minor_breaks = NULL, breaks = seq(-1, 1, by = 0.2))

### Figure 6: accuracy of the minimal models as a function of $r$

Average accuracy of $M1$–$M3$ (LDA pairwise classifiers, 1000 repetitions for each of 50 values of $r$), with 95% confidence bands. The simulation runs in parallel and takes a few minutes, so by default the stored result `data/example2_lda.csv` is used; set `RECOMPUTE_EXAMPLE2 <- TRUE` to rerun it (the seed is that of the paper).

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 4)
RECOMPUTE_EXAMPLE2 <- FALSE
exm2 <- if (RECOMPUTE_EXAMPLE2) lda_example2(reps = 1000) else read.csv("data/example2_lda.csv")
exm2_cols <- c(M1 = "#619CFF", M2 = "#00BA38", M3 = "#F8766D")
exm2_labs <- c(M1 = "M1 (omits C2-C3)", M2 = "M2 (omits C1-C3)", M3 = "M3 (omits C1-C2)")
ggplot(exm2, aes(x = r, y = accuracy, colour = model, fill = model)) +
  geom_ribbon(aes(ymin = accuracy - 1.96 * se, ymax = accuracy + 1.96 * se), alpha = 0.2, colour = NA) +
  geom_line(aes(linetype = model), linewidth = 0.7) +
  theme_minimal(base_size = 10) +
  labs(x = expression(plain(correlation)~italic(r)), y = "average accuracy",
       colour = "model", fill = "model", linetype = "model") +
  scale_colour_manual(values = exm2_cols, labels = exm2_labs, aesthetics = c("colour", "fill")) +
  scale_linetype_manual(values = c(M1 = "solid", M2 = "dashed", M3 = "dotdash"), labels = exm2_labs) +
  theme(legend.key.width = grid::unit(1.4, "cm")) +
  scale_x_continuous(minor_breaks = NULL, breaks = seq(-0.8, 0.8, by = 0.2))

### Table 1: seven von Mises–Fisher classes, by tree diameter

Test accuracy of minimal models for seven von Mises–Fisher classes centred on the optimal Tammes configuration ($\kappa = 5$, LDA pairwise classifiers, 20 runs), grouped by the diameter of the spanning tree. *trees in K7*: the number of spanning trees of each diameter; *n*: the number of evaluated minimal models.

In [ ]:
exm3_means <- tammes_means(7)
exm3_trees <- candidate_trees(7, most_distant_pairs(exm3_means))
exm3 <- run_experiment(kappa = 5, means = exm3_means, trees = exm3_trees,
                       classifier = "lda", n_train = 100, n_test = 200,
                       reps = 20, max_per_dia = 50, seed = 123)
exm3_tab <- do.call(rbind, lapply(split(exm3$acc, exm3$dia), function(a)
  data.frame(mean = mean(a), median = median(a), sd = sd(a), n = length(a))))
exm3_tab <- data.frame(diameter = as.integer(rownames(exm3_tab)),
                  `trees in K7` = as.integer(table(exm3_trees$dia)[rownames(exm3_tab)]),
                  exm3_tab, check.names = FALSE)
rownames(exm3_tab) <- NULL
cat(sprintf("Bayes accuracy: %.3f\n", mean(exm3$bayes_acc)))
mutate(exm3_tab, across(c(mean, median, sd), ~ round(.x, 3)))

### Table 2: ten von Mises–Fisher classes, diameter-3 trees by split

Test accuracy of minimal models of diameter-3 trees for ten von Mises–Fisher classes centred on the optimal Tammes configuration ($\kappa = 10.5$, LDA pairwise classifiers, 20 runs), grouped by the split $\{p, q\}$ of the double star. *trees in K10*: the number of spanning trees with each split; *n*: the number of evaluated minimal models.

In [ ]:
exm5 <- run_diam3_experiment(kappa = 10.5, means = tammes_means(10), classifier = "lda",
                             n_train = 100, n_test = 200, reps = 20, per_split = 50, seed = 123)
exm5_splits <- diam3_splits(10)
exm5_tab <- do.call(rbind, lapply(split(exm5$acc, exm5$p), function(a)
  data.frame(mean = mean(a), median = median(a), sd = sd(a), n = length(a))))
exm5_tab <- data.frame(split = sprintf("{%d, %d}", exm5_splits$p, exm5_splits$q),
                  Omega3 = as.integer(exm5_splits$omega3),
                  `average path length` = exm5_splits$apl,
                  `trees in K10` = as.integer(exm5_splits$trees),
                  exm5_tab, check.names = FALSE)
rownames(exm5_tab) <- NULL
cat(sprintf("Bayes accuracy: %.3f\n", mean(exm5$bayes_acc)))
mutate(exm5_tab, across(c(`average path length`, mean, median, sd), ~ round(.x, 3)))

## Section 5: Algorithms A and B, and all spanning trees

The results of the RBF-SVM experiments are read from `data/` (20 runs; the trees are selected by 10-fold cross-validation). The files were produced by `experiments/run_svm_k3.R` (`alg1svm-*`) and by `eval3_all(..., file = "alg2")` of `experiments/exp4.R` (`alg2svm-*`).

### Figure 7: three-class datasets

Test error, Brier score and cross-entropy of Algorithms A and B and of the full one-vs-one model with the coupling of Wu, Lin and Weng (WLW) on the three-class datasets. Light green: the better of A and B by mean, and the other one if it is not significantly different from it (paired $t$-test over the 20 runs, $\alpha = 0.05$); blue: the baseline WLW.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5.2)
k3 <- map_dfr(c("dna", "waveform"), ~ read.csv(sprintf("data/alg1svm-%s-1.csv", .x))) |>
  filter(cv == 10 | alg == "WLW") |>
  group_by(dataset, metric, run, alg) |>
  summarise(score = mean(test_score), .groups = "drop") |>
  mutate(alg = factor(alg, levels = c("A", "B", "WLW")), metric = metric_factor(metric))
k3 <- k3 |> group_by(dataset, metric) |>
  group_modify(~ best_group(.x, among = c("A", "B"))) |> ungroup()
ggplot(k3, aes(alg, score, fill = grp)) +
  geom_boxplot(outlier.size = 0.8, linewidth = 0.4) +
  facet_wrap(vars(metric, dataset), ncol = 2, scales = "free_y",
             labeller = function(l) label_value(l, multi_line = FALSE)) +
  baseline_fill + labs(x = NULL, y = NULL) + theme_minimal() +
  baseline_axis(levels(k3$alg))

### Figure 8: comparison with the best spanning tree (TR)

Difference in test error, Brier score and cross-entropy between Algorithms A, B, BR, D3 or WLW and Algorithm TR, the best of all spanning trees, on *satimage* ($K=6$) and *segment* ($K=7$); differences within a run, negative values are better than TR. Light green: the best of A, B, BR, D3 and TR by mean and the algorithms not significantly different from it; TR itself is the zero line. Blue: the baseline WLW.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 5.2)
dss2 <- c("satimage", "segment")
k67 <- map_dfr(dss2, ~ read.csv(sprintf("data/alg2svm-%s-1.csv", .x))) |>
  filter(cv == 10 | alg == "WLW") |>
  group_by(dataset, metric, run, alg) |>
  summarise(score = mean(test_score), .groups = "drop") |>
  mutate(dataset = factor(dataset, levels = dss2),
         alg = factor(alg, levels = c("A", "B", "BR", "D3", "TR", "WLW")),
         metric = metric_factor(metric))
grp67 <- k67 |> group_by(dataset, metric) |>
  group_modify(~ best_group(.x, among = c("A", "B", "BR", "D3", "TR"))) |> ungroup() |>
  dplyr::select(dataset, metric, run, alg, grp)
k67d <- k67 |> group_by(dataset, metric, run) |>
  mutate(diff = score - score[alg == "TR"]) |> ungroup() |> filter(alg != "TR") |>
  left_join(grp67, by = c("dataset", "metric", "run", "alg"))
ggplot(k67d, aes(alg, diff, fill = grp)) +
  geom_hline(yintercept = 0, colour = "grey40", linewidth = 0.4) +
  geom_boxplot(outlier.size = 0.8, linewidth = 0.4) +
  facet_wrap(vars(metric, dataset), ncol = 2, scales = "free_y",
             labeller = function(l) label_value(l, multi_line = FALSE)) +
  baseline_fill + labs(x = NULL, y = NULL) + theme_minimal() +
  baseline_axis(setdiff(levels(k67d$alg), "TR"))

### Table 3: mean scores on *satimage* and *segment*

Mean test error, Brier score and cross-entropy over the 20 runs. The trees of A, B, BR, D3 and TR are selected by 10-fold cross-validation from the number of candidates given in the column *candidates*.

In [ ]:
# BR: brooms B_(K-2,2); D3: the double stars plus the K stars; TR: all K^(K-2) trees
ncand <- function(K) c(A = 1, B = K, BR = K * (K - 1) * (K - 2),
                       D3 = choose(K, 2) * (2^(K - 2) - 2) + K, TR = K^(K - 2), WLW = NA)
k67 |> group_by(dataset, alg, metric) |> summarise(score = mean(score), .groups = "drop") |>
  pivot_wider(names_from = metric, values_from = score) |>
  mutate(candidates = ifelse(dataset == "satimage", ncand(6)[as.character(alg)], ncand(7)[as.character(alg)])) |>
  arrange(dataset, alg) |>
  transmute(dataset = as.character(dataset), algorithm = as.character(alg),
            candidates = ifelse(is.na(candidates), "", format(candidates, big.mark = ",", trim = TRUE)),
            `test error` = sprintf("%.4f", `test error`), `Brier score` = sprintf("%.4f", `Brier score`),
            `cross-entropy` = sprintf("%.4f", `cross-entropy`))

## Section 6: Algorithms A–G on five datasets

Results of `experiments/eval_svm_alg3.R` (20 runs, 7 repetitions of 10-fold cross-validation; the scores of a run are averaged over the 7 repetitions) and of `experiments/dagsvm_svm.R` (DAGSVM, evaluated on the same pairwise SVMs).

In [ ]:
dss3 <- c("satimage", "segment", "usps", "mnist", "letter")
K3 <- c(satimage = 6, segment = 7, usps = 10, mnist = 10, letter = 26)
alg3 <- map_dfr(sprintf("data/alg3svm-%s-7.csv", dss3), read.csv) |>
  group_by(dataset, metric, run, alg) |>
  summarise(score = mean(test_score), .groups = "drop") |>
  mutate(dataset = factor(dataset, levels = dss3),
         alg = factor(alg, levels = c(LETTERS[1:7], "WLW")),
         metric = metric_factor(metric))

### Table 5: mean scores of Algorithms A–G, DAGSVM and WLW

Mean test error (3 decimals), Brier score and cross-entropy (2 decimals) over the 20 runs. In the paper the best of Algorithms A–G is set in bold; here it is shown in brackets, followed by `*` if it is significantly better than the runner-up among A–G (paired $t$-test over the 20 runs, $p < 0.05$). DAGSVM and WLW are references and are not marked.

In [ ]:
# paired t-test (over runs) of the best of A-G against the runner-up
sig3 <- alg3 |> filter(alg != "WLW") |> group_by(dataset, metric) |>
  group_modify(function(d, key) {
    o <- d |> group_by(alg) |> summarise(m = mean(score), .groups = "drop") |> arrange(m)
    s1 <- d |> filter(alg == o$alg[1]) |> arrange(run); s2 <- d |> filter(alg == o$alg[2]) |> arrange(run)
    tibble(p = t.test(s1$score, s2$score, paired = TRUE)$p.value)
  }) |> ungroup()
fmt3 <- function(x, metric) sub("^0[.]", ".", sprintf(ifelse(metric == "test error", "%.3f", "%.2f"), x))
tab3 <- alg3 |> group_by(dataset, metric, alg) |> summarise(mean = mean(score), .groups = "drop") |>
  left_join(sig3, by = c("dataset", "metric")) |> group_by(dataset, metric) |>
  mutate(best = alg != "WLW" & mean == min(mean[alg != "WLW"])) |> ungroup() |>
  mutate(alg = as.character(alg), txt = fmt3(mean, metric),
         txt = ifelse(best, paste0("[", txt, "]", ifelse(p < 0.05, "*", "")), txt))
dag3 <- read.csv("data/dagsvm-svm.csv") |> filter(method == "DAG") |>
  group_by(dataset, metric) |> summarise(mean = mean(score), .groups = "drop") |>
  mutate(alg = "DAG", metric = metric_factor(metric), dataset = factor(dataset, levels = dss3),
         txt = fmt3(mean, metric))
bind_rows(tab3, dag3) |>
  mutate(dataset = factor(paste0(dataset, " (K=", K3[as.character(dataset)], ")"),
                          levels = paste0(dss3, " (K=", K3, ")"))) |>
  dplyr::select(metric, alg, dataset, txt) |>
  pivot_wider(names_from = dataset, values_from = txt, names_sort = TRUE) |>
  arrange(metric, match(alg, c(LETTERS[1:7], "DAG", "WLW"))) |>
  as.data.frame()

### Figure 9: Algorithms A–G and WLW on the five datasets

Each observation is the mean over the 7 repetitions of cross-validation of one run. Light green: the best of Algorithms A–G by mean and the algorithms not significantly different from it (paired $t$-test over the 20 runs, $\alpha = 0.05$); blue: WLW.

In [ ]:
options(repr.plot.width = 7, repr.plot.height = 9)
alg3f <- alg3 |> group_by(dataset, metric) |>
  group_modify(~ best_group(.x, among = LETTERS[1:7])) |> ungroup()
# one panel per dataset (row) and metric (column), each with its own y scale
mets3 <- levels(alg3f$metric); dsl3 <- levels(alg3f$dataset)
panels3 <- list()
for (ds in dsl3) for (mt in mets3) {
  p <- ggplot(alg3f |> filter(dataset == ds, metric == mt), aes(alg, score, fill = grp)) +
    geom_boxplot(outlier.size = 0.6, linewidth = 0.35) +
    baseline_fill + theme_minimal(base_size = 9) + baseline_axis(levels(alg3$alg)) +
    labs(x = NULL, y = if (mt == mets3[1]) ds else NULL, title = if (ds == dsl3[1]) mt else NULL) +
    theme(axis.text = element_text(size = 8.5),
          axis.title.y = element_text(size = 10.5, face = "bold"),
          plot.title = element_text(size = 10.5, face = "bold", hjust = 0.5))
  if (ds != tail(dsl3, 1)) p <- p + theme(axis.text.x = element_blank())
  panels3[[length(panels3) + 1]] <- p
}
wrap_plots(panels3, ncol = 3)

## Section 7: recalibration of minimal models

Results of `experiments/recalib_svm_alg3.R`, computed from the pairwise SVMs of Section 6 (method `U`: the uncalibrated selection of Section 6; `R`: selection among recalibrated trees).

### Table 6: recalibrated Algorithms C and G

*recal.*: the mean score of the recalibrated model; *diff*: the remaining difference to WLW, recal. − WLW; *impr.*: the improvement, the part of the difference to WLW closed by recalibration, (uncal. − recal.) / (uncal. − WLW), where uncal. is the score without recalibration (Table 5). The rows *average* give the mean improvement over the five datasets.

In [ ]:
recal <- map_dfr(sprintf("data/recal-svm-%s-7.csv", dss3), read.csv) |>
  group_by(dataset, metric, run, alg, method) |>
  summarise(score = mean(test_score), .groups = "drop") |>
  mutate(dataset = factor(dataset, levels = dss3), metric = metric_factor(metric))
rm_ <- recal |> filter(alg %in% c("C", "G", "WLW")) |>
  group_by(dataset, metric, alg, method) |> summarise(mean = mean(score), .groups = "drop")
num <- function(x) sub("^0[.]", ".", sprintf("%.3f", x))
pct <- function(x) sprintf("%.0f%%", 100 * x)
w6 <- rm_ |> filter(alg != "WLW") |> pivot_wider(names_from = method, values_from = mean) |>
  left_join(rm_ |> filter(alg == "WLW") |> dplyr::select(dataset, metric, WLW = mean), by = c("dataset", "metric")) |>
  mutate(impr = (U - R) / (U - WLW))
rows6 <- w6 |>
  transmute(metric, dataset = as.character(dataset), K = K3[as.character(dataset)], alg,
            recal. = num(R), diff = num(R - WLW), impr. = pct(impr), WLW = num(WLW)) |>
  pivot_wider(names_from = alg, values_from = c(recal., diff, impr.), names_glue = "{alg} {.value}")
avg6 <- w6 |> group_by(metric, alg) |> summarise(impr = mean(impr), .groups = "drop") |>
  transmute(metric, dataset = "average", alg, impr. = pct(impr)) |>
  pivot_wider(names_from = alg, values_from = impr., names_glue = "{alg} impr.")
bind_rows(rows6, avg6) |>
  arrange(metric, dataset == "average") |>
  dplyr::select(metric, dataset, K, `C recal.`, `C diff`, `C impr.`, `G recal.`, `G diff`, `G impr.`, WLW) |>
  mutate(across(everything(), ~ ifelse(is.na(.x), "", as.character(.x)))) |>
  as.data.frame()